# DIP prject (SROOE)

## -1. Controling variables

In [1]:
PORTION = 0.025      # How much of the dataset to use for training. A number between 0.0 and 1.0 to determine how much of the data being randomly sampled and used for training.
N_ITER = 300        # Number of iteration used during training (i.e. total_iters). This together with train_size (which is equal to ceil(dataset_size / batch_size)) determine the number of total_epochs for training. (total_epochs = ceil(total_iters / train_size))
VAL_FREQUENCY = 100  # How often the validation must be run during traininig.

## 0. Loading files into kaggle working dir

In [2]:
!ls /kaggle/input/

df2kdata  modified-files  srooe-proj  srot-chck-pth  srot-proj


In [3]:
!cp -r /kaggle/input/srooe-proj /kaggle/input/srot-chck-pth /kaggle/input/srot-proj /kaggle/working/

In [4]:
!ls /kaggle/working/

srooe-proj  srot-chck-pth  srot-proj


In [5]:
!ls /kaggle/input/modified-files/modified-files

generate_T_OOS_Map.py  lpips_measure.py  LQGT_dataset.py  train.py


In [6]:
!cp /kaggle/input/modified-files/modified-files/train.py /kaggle/working/srooe-proj/SROOE-main/codes/

In [7]:
!cp /kaggle/input/modified-files/modified-files/LQGT_dataset.py /kaggle/working/srooe-proj/SROOE-main/codes/data/

In [8]:
!cp /kaggle/input/modified-files/modified-files/generate_T_OOS_Map.py /kaggle/working/srot-proj/SROT-main/LPIPS-Map-Gen/

In [9]:
!cp /kaggle/input/modified-files/modified-files/lpips_measure.py /kaggle/working/srot-proj/SROT-main/LPIPS-Map-Gen/

# Using pre-trained SROT model to compute T_OOS_Maps for training SROOE

## 1. Downloading dataset and modifying test.yml file of SROT

### Loading test.yml file of SROT

In [10]:
import yaml
import pprint

# Replace 'your_file.yaml' with the name of your uploaded YAML file
srot_file_path = '/kaggle/working/srot-proj/SROT-main/codes/options/test/test.yml'

# Read and print the YAML content
with open(srot_file_path, 'r') as f:
    srot_test_data = yaml.safe_load(f)
    print('Test yaml file of SROT project:\n')
    pprint.pprint(srot_test_data)

Test yaml file of SROT project:

{'datasets': {'test_100': {'dataroot_LQ': 'E:\\exp\\dataset\\DIV2K_valid_LR_bicubic\\X4',
                           'mode': 'LQ',
                           'name': 'DIV2K_val_Q100'}},
 'distortion': 'sr',
 'gpu_ids': [0],
 'model': 'srgan',
 'name': 'ESRGAN-SROT-M1234-v2-4x',
 'network_G': {'in_nc': 3,
               'nb': 23,
               'nf': 64,
               'out_nc': 3,
               'upscale': 4,
               'which_model_G': 'RRDBNet'},
 'path': {'pretrain_model_G': 'E:\\github\\SROT-main\\pretrained/ESRGAN-SROT-M1234-v2-4x.pth'},
 'scale': 4,
 'suffix': None}


### Setting the path to pre-trained SROT model

In [11]:
pre_trained_SROT_path = '/kaggle/working/srot-chck-pth/SR.pth'

In [12]:
srot_test_data['path']['pretrain_model_G'] = pre_trained_SROT_path
print(f"The path has been set to {srot_test_data['path']['pretrain_model_G']}")

The path has been set to /kaggle/working/srot-chck-pth/SR.pth


### Modifying SROT test.yml file as stated in SROOE README.md file

#### Downloading 800 Low Res images

In [13]:
# Fetch the DIV2K ×4 LR zip
!wget -q https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_LR_bicubic_X4.zip

In [14]:
!mkdir -p DIV2K_train_LRx4

In [15]:
!ls '/kaggle/working/'

DIV2K_train_LR_bicubic_X4.zip  srooe-proj     srot-proj
DIV2K_train_LRx4	       srot-chck-pth


In [16]:
# Unzip into a clean folder
!unzip -q DIV2K_train_LR_bicubic_X4.zip -d DIV2K_train_LRx4

In [17]:
!ls '/kaggle/working/DIV2K_train_LRx4/DIV2K_train_LR_bicubic/X4'

0001x4.png  0135x4.png	0269x4.png  0403x4.png	0537x4.png  0671x4.png
0002x4.png  0136x4.png	0270x4.png  0404x4.png	0538x4.png  0672x4.png
0003x4.png  0137x4.png	0271x4.png  0405x4.png	0539x4.png  0673x4.png
0004x4.png  0138x4.png	0272x4.png  0406x4.png	0540x4.png  0674x4.png
0005x4.png  0139x4.png	0273x4.png  0407x4.png	0541x4.png  0675x4.png
0006x4.png  0140x4.png	0274x4.png  0408x4.png	0542x4.png  0676x4.png
0007x4.png  0141x4.png	0275x4.png  0409x4.png	0543x4.png  0677x4.png
0008x4.png  0142x4.png	0276x4.png  0410x4.png	0544x4.png  0678x4.png
0009x4.png  0143x4.png	0277x4.png  0411x4.png	0545x4.png  0679x4.png
0010x4.png  0144x4.png	0278x4.png  0412x4.png	0546x4.png  0680x4.png
0011x4.png  0145x4.png	0279x4.png  0413x4.png	0547x4.png  0681x4.png
0012x4.png  0146x4.png	0280x4.png  0414x4.png	0548x4.png  0682x4.png
0013x4.png  0147x4.png	0281x4.png  0415x4.png	0549x4.png  0683x4.png
0014x4.png  0148x4.png	0282x4.png  0416x4.png	0550x4.png  0684x4.png
0015x4.png  0149x4.png	0283x4.png 

#### Downloading 800 High Res images

In [18]:
!wget -q https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip

In [19]:
!mkdir -p DIV2K_train_HR

In [20]:
!ls '/kaggle/working/'

DIV2K_train_HR	    DIV2K_train_LR_bicubic_X4.zip  srooe-proj	  srot-proj
DIV2K_train_HR.zip  DIV2K_train_LRx4		   srot-chck-pth


In [21]:
# Unzip into a clean folder
!unzip -q DIV2K_train_HR.zip -d DIV2K_train_HR

In [22]:
!ls '/kaggle/working/DIV2K_train_HR/DIV2K_train_HR'

0001.png  0101.png  0201.png  0301.png	0401.png  0501.png  0601.png  0701.png
0002.png  0102.png  0202.png  0302.png	0402.png  0502.png  0602.png  0702.png
0003.png  0103.png  0203.png  0303.png	0403.png  0503.png  0603.png  0703.png
0004.png  0104.png  0204.png  0304.png	0404.png  0504.png  0604.png  0704.png
0005.png  0105.png  0205.png  0305.png	0405.png  0505.png  0605.png  0705.png
0006.png  0106.png  0206.png  0306.png	0406.png  0506.png  0606.png  0706.png
0007.png  0107.png  0207.png  0307.png	0407.png  0507.png  0607.png  0707.png
0008.png  0108.png  0208.png  0308.png	0408.png  0508.png  0608.png  0708.png
0009.png  0109.png  0209.png  0309.png	0409.png  0509.png  0609.png  0709.png
0010.png  0110.png  0210.png  0310.png	0410.png  0510.png  0610.png  0710.png
0011.png  0111.png  0211.png  0311.png	0411.png  0511.png  0611.png  0711.png
0012.png  0112.png  0212.png  0312.png	0412.png  0512.png  0612.png  0712.png
0013.png  0113.png  0213.png  0313.png	0413.png  0513.png  0613.

### Using a portion of the data just to check if the code runs as expected

In [23]:
import os, random, shutil

# Paths
LR_DIR     = '/kaggle/working/DIV2K_train_LRx4/DIV2K_train_LR_bicubic/X4'
SAMPLE_DIR = "SAMPLE/DIV2K_train_LR_bicubic_X4_sample"

# Gather all .png files
all_fns = [fn for fn in os.listdir(LR_DIR) if fn.endswith(".png")]

# Sample 10%
random.seed(42)  # for reproducibility!
n_sample = int(len(all_fns) * PORTION)
sample_fns = random.sample(all_fns, n_sample)

# Create output directory
os.makedirs(SAMPLE_DIR, exist_ok=True)

# Copy sampled files over
for fn in sample_fns:
    src = os.path.join(LR_DIR, fn)
    dst = os.path.join(SAMPLE_DIR, fn)
    shutil.copy(src, dst)

print(f"Sampled {n_sample} images into '{SAMPLE_DIR}/'")


Sampled 20 images into 'SAMPLE/DIV2K_train_LR_bicubic_X4_sample/'


In [24]:
HR_DIR     = '/kaggle/working/DIV2K_train_HR/DIV2K_train_HR'
HR_SAMPLE  = "SAMPLE/DIV2K_train_HR_sample"
os.makedirs(HR_SAMPLE, exist_ok=True)

for fn in sample_fns:
    removed_4x_fn = fn.replace('x4', '')
    shutil.copy(os.path.join(HR_DIR, removed_4x_fn), os.path.join(HR_SAMPLE, removed_4x_fn))


#### Changing the name and path to dataroot_LQ

In [25]:
srot_test_data['datasets']['test_100']['dataroot_LQ'] = '/kaggle/working/SAMPLE/DIV2K_train_LR_bicubic_X4_sample'
srot_test_data['datasets']['test_100']['name'] = 'DIV2K_train_HR'

### Dumping the data into SROT test.yml file

In [26]:
with open(srot_file_path, 'w') as f:
    yaml.dump(srot_test_data, f, sort_keys=False)
    print('Test yaml file of SROT project updated:\n')
    pprint.pprint(srot_test_data)

Test yaml file of SROT project updated:

{'datasets': {'test_100': {'dataroot_LQ': '/kaggle/working/SAMPLE/DIV2K_train_LR_bicubic_X4_sample',
                           'mode': 'LQ',
                           'name': 'DIV2K_train_HR'}},
 'distortion': 'sr',
 'gpu_ids': [0],
 'model': 'srgan',
 'name': 'ESRGAN-SROT-M1234-v2-4x',
 'network_G': {'in_nc': 3,
               'nb': 23,
               'nf': 64,
               'out_nc': 3,
               'upscale': 4,
               'which_model_G': 'RRDBNet'},
 'path': {'pretrain_model_G': '/kaggle/working/srot-chck-pth/SR.pth'},
 'scale': 4,
 'suffix': None}


## 2. Generating SROT results

In [27]:
cd ./srot-proj/SROT-main/codes

/kaggle/working/srot-proj/SROT-main/codes


In [28]:
!ls

data  models  options  scripts	test.py  train.py  utils


In [29]:
!ls -lh /kaggle/working


total 3.6G
drwxr-xr-x 3 root root 4.0K Aug  9 04:54 DIV2K_train_HR
-rw-r--r-- 1 root root 3.3G Feb 14  2017 DIV2K_train_HR.zip
-rw-r--r-- 1 root root 236M Feb 14  2017 DIV2K_train_LR_bicubic_X4.zip
drwxr-xr-x 3 root root 4.0K Aug  9 04:51 DIV2K_train_LRx4
drwxr-xr-x 4 root root 4.0K Aug  9 04:55 SAMPLE
drwxr-xr-x 3 root root 4.0K Aug  9 04:51 srooe-proj
drwxr-xr-x 2 root root 4.0K Aug  9 04:51 srot-chck-pth
drwxr-xr-x 3 root root 4.0K Aug  9 04:51 srot-proj


In [30]:
!ls -l /kaggle/working/srot-chck-pth


total 72704
-rw-r--r-- 1 root root 74447247 Aug  9 04:51 SR.pth


In [31]:
import subprocess

for i in range(0, 101, 10):
    t = i / 100
    t_str = f"{t:.2f}"
    print(f"→ threshold = {t_str}")
    subprocess.run(["pwd"], check=True)
    subprocess.run(
        ["python", "test.py", "-opt", "options/test/test.yml", "-t", t_str],
        check=True
    )


→ threshold = 0.00
/kaggle/working/srot-proj/SROT-main/codes


25-08-09 04:55:13.928 - INFO:   name: ESRGAN-SROT-M1234-v2-4x_t000
  suffix: None
  model: srgan
  distortion: sr
  scale: 4
  gpu_ids: [0]
  datasets:[
    test_100:[
      name: DIV2K_train_HR
      mode: LQ
      dataroot_LQ: /kaggle/working/SAMPLE/DIV2K_train_LR_bicubic_X4_sample
      phase: test
      scale: 4
      data_type: img
    ]
  ]
  network_G:[
    which_model_G: RRDBNet
    in_nc: 3
    out_nc: 3
    nf: 64
    nb: 23
    upscale: 4
    scale: 4
  ]
  path:[
    pretrain_model_G: /kaggle/working/srot-chck-pth/SR.pth
    root: /kaggle/working/srot-proj/SROT-main
    results_root: /kaggle/working/srot-proj/SROT-main/results/ESRGAN-SROT-M1234-v2-4x_t000
    log: /kaggle/working/srot-proj/SROT-main/results/ESRGAN-SROT-M1234-v2-4x_t000
  ]
  is_train: False

25-08-09 04:55:13.933 - INFO: Dataset [LQDataset - DIV2K_train_HR] is created.
25-08-09 04:55:13.933 - INFO: Number of test images in [DIV2K_train_HR]: 20
25-08-09 04:55:14.970 - INFO: Network G structure: DataParallel 

export CUDA_VISIBLE_DEVICES=0


25-08-09 04:55:17.682 - INFO: 0020x4              
25-08-09 04:55:19.121 - INFO: 0040x4              
25-08-09 04:55:20.590 - INFO: 0074x4              
25-08-09 04:55:21.958 - INFO: 0077x4              
25-08-09 04:55:23.380 - INFO: 0214x4              
25-08-09 04:55:24.799 - INFO: 0256x4              
25-08-09 04:55:26.188 - INFO: 0267x4              
25-08-09 04:55:27.573 - INFO: 0314x4              
25-08-09 04:55:28.978 - INFO: 0328x4              
25-08-09 04:55:30.669 - INFO: 0353x4              
25-08-09 04:55:32.086 - INFO: 0389x4              
25-08-09 04:55:33.529 - INFO: 0483x4              
25-08-09 04:55:34.937 - INFO: 0526x4              
25-08-09 04:55:36.246 - INFO: 0534x4              
25-08-09 04:55:37.710 - INFO: 0539x4              
25-08-09 04:55:39.169 - INFO: 0558x4              
25-08-09 04:55:40.426 - INFO: 0583x4              
25-08-09 04:55:41.421 - INFO: 0710x4              
25-08-09 04:55:42.835 - INFO: 0746x4              
25-08-09 04:55:44.404 - INFO: 0

→ threshold = 0.10
/kaggle/working/srot-proj/SROT-main/codes


...

25-08-09 05:00:18.532 - INFO: 0020x4              
25-08-09 05:00:20.006 - INFO: 0040x4              
25-08-09 05:00:21.456 - INFO: 0074x4              
25-08-09 05:00:22.857 - INFO: 0077x4              
25-08-09 05:00:24.303 - INFO: 0214x4              
25-08-09 05:00:25.719 - INFO: 0256x4              
25-08-09 05:00:27.132 - INFO: 0267x4              
25-08-09 05:00:28.588 - INFO: 0314x4              
25-08-09 05:00:30.040 - INFO: 0328x4              
25-08-09 05:00:31.651 - INFO: 0353x4              
25-08-09 05:00:33.073 - INFO: 0389x4              
25-08-09 05:00:34.501 - INFO: 0483x4              
25-08-09 05:00:35.918 - INFO: 0526x4              
25-08-09 05:00:37.205 - INFO: 0534x4              
25-08-09 05:00:38.626 - INFO: 0539x4              
25-08-09 05:00:40.044 - INFO: 0558x4              
25-08-09 05:00:41.296 - INFO: 0583x4              
25-08-09 05:00:42.292 - INFO: 0710x4              
25-08-09 05:00:43.695 - INFO: 0746x4              
25-08-09 05:00:45.315 - INFO: 0

→ threshold = 1.00
/kaggle/working/srot-proj/SROT-main/codes


25-08-09 05:00:48.877 - INFO:   name: ESRGAN-SROT-M1234-v2-4x_t100
  suffix: None
  model: srgan
  distortion: sr
  scale: 4
  gpu_ids: [0]
  datasets:[
    test_100:[
      name: DIV2K_train_HR
      mode: LQ
      dataroot_LQ: /kaggle/working/SAMPLE/DIV2K_train_LR_bicubic_X4_sample
      phase: test
      scale: 4
      data_type: img
    ]
  ]
  network_G:[
    which_model_G: RRDBNet
    in_nc: 3
    out_nc: 3
    nf: 64
    nb: 23
    upscale: 4
    scale: 4
  ]
  path:[
    pretrain_model_G: /kaggle/working/srot-chck-pth/SR.pth
    root: /kaggle/working/srot-proj/SROT-main
    results_root: /kaggle/working/srot-proj/SROT-main/results/ESRGAN-SROT-M1234-v2-4x_t100
    log: /kaggle/working/srot-proj/SROT-main/results/ESRGAN-SROT-M1234-v2-4x_t100
  ]
  is_train: False

25-08-09 05:00:48.961 - INFO: Dataset [LQDataset - DIV2K_train_HR] is created.
25-08-09 05:00:48.961 - INFO: Number of test images in [DIV2K_train_HR]: 20
25-08-09 05:00:49.672 - INFO: Network G structure: DataParallel 

export CUDA_VISIBLE_DEVICES=0


25-08-09 05:00:51.887 - INFO: 0020x4              
25-08-09 05:00:53.363 - INFO: 0040x4              
25-08-09 05:00:54.815 - INFO: 0074x4              
25-08-09 05:00:56.263 - INFO: 0077x4              
25-08-09 05:00:57.698 - INFO: 0214x4              
25-08-09 05:00:59.131 - INFO: 0256x4              
25-08-09 05:01:00.561 - INFO: 0267x4              
25-08-09 05:01:01.981 - INFO: 0314x4              
25-08-09 05:01:03.394 - INFO: 0328x4              
25-08-09 05:01:05.028 - INFO: 0353x4              
25-08-09 05:01:06.442 - INFO: 0389x4              
25-08-09 05:01:07.897 - INFO: 0483x4              
25-08-09 05:01:09.318 - INFO: 0526x4              
25-08-09 05:01:10.612 - INFO: 0534x4              
25-08-09 05:01:12.086 - INFO: 0539x4              
25-08-09 05:01:13.525 - INFO: 0558x4              
25-08-09 05:01:14.787 - INFO: 0583x4              
25-08-09 05:01:15.794 - INFO: 0710x4              
25-08-09 05:01:17.220 - INFO: 0746x4              
25-08-09 05:01:18.799 - INFO: 0

## 3. Generating LPIPS maps for the SROT results

In [32]:
cd ../LPIPS-Map-Gen

/kaggle/working/srot-proj/SROT-main/LPIPS-Map-Gen


In [33]:
!ls

generate_T_OOS_Map.py  lpips_measure.py  util.py  utils.py


In [34]:
print(sample_fns)

['0539x4.png', '0526x4.png', '0040x4.png', '0267x4.png', '0534x4.png', '0077x4.png', '0583x4.png', '0710x4.png', '0020x4.png', '0074x4.png', '0483x4.png', '0214x4.png', '0777x4.png', '0389x4.png', '0256x4.png', '0558x4.png', '0328x4.png', '0353x4.png', '0746x4.png', '0314x4.png']


In [35]:
# Install the PyPI lpips package
!pip install --quiet lpips


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 55.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 81.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 16.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 81.3 MB/s eta 0:00:00:00:0100:01


#### Work-around for name conflict between grount truth and resolved images. Must be removed after code refactoring

In [36]:
from pathlib import Path

base_res = Path("/kaggle/working/srot-proj/SROT-main/results/ESRGAN-SROT-M1234-v2-4x_t")

for i in range(0, 101, 10):
    t3     = f"{i:03d}"
    sr_dir = Path(f"{base_res}{t3}/DIV2K_train_HR")
    if not sr_dir.exists():
        continue

    print(f"Renaming in {sr_dir}…")
    for p in sr_dir.glob("*.png"):
        stem = p.stem

        # adjust this if your suffix uses underscore: e.g. "_4x"
        if stem.endswith("x4"):
            new_name = stem[:-2] + p.suffix  # drop the 'x4'
            p.rename(p.with_name(new_name))


Renaming in /kaggle/working/srot-proj/SROT-main/results/ESRGAN-SROT-M1234-v2-4x_t000/DIV2K_train_HR…
Renaming in /kaggle/working/srot-proj/SROT-main/results/ESRGAN-SROT-M1234-v2-4x_t010/DIV2K_train_HR…
Renaming in /kaggle/working/srot-proj/SROT-main/results/ESRGAN-SROT-M1234-v2-4x_t020/DIV2K_train_HR…
Renaming in /kaggle/working/srot-proj/SROT-main/results/ESRGAN-SROT-M1234-v2-4x_t030/DIV2K_train_HR…
Renaming in /kaggle/working/srot-proj/SROT-main/results/ESRGAN-SROT-M1234-v2-4x_t040/DIV2K_train_HR…
Renaming in /kaggle/working/srot-proj/SROT-main/results/ESRGAN-SROT-M1234-v2-4x_t050/DIV2K_train_HR…
Renaming in /kaggle/working/srot-proj/SROT-main/results/ESRGAN-SROT-M1234-v2-4x_t060/DIV2K_train_HR…
Renaming in /kaggle/working/srot-proj/SROT-main/results/ESRGAN-SROT-M1234-v2-4x_t070/DIV2K_train_HR…
Renaming in /kaggle/working/srot-proj/SROT-main/results/ESRGAN-SROT-M1234-v2-4x_t080/DIV2K_train_HR…
Renaming in /kaggle/working/srot-proj/SROT-main/results/ESRGAN-SROT-M1234-v2-4x_t090/DIV2K_

#### Generating LPIPS maps

In [37]:
import subprocess
from pathlib import Path

# Paths (adjust these as needed)
gt_dir   = Path("/kaggle/working/SAMPLE/DIV2K_train_HR_sample") #Path("path_to_GT/DIV2K_train_HR")
base_res = Path("/kaggle/working/srot-proj/SROT-main/results/ESRGAN-SROT-M1234-v2-4x_t") # Path("path_to_SROT/SROT-main/results/ESRGAN-SROT-M1234-v2-4x")

for i in range(0, 101, 10):
    # format i=0 → "000", …, 100 → "100"
    t3 = f"{i:03d}"
    sr_dir = Path(f"{base_res}{t3}/DIV2K_train_HR")

    print(f"\n=== Threshold t{t3} ===")
    print("GT dir:", gt_dir)
    print("SR dir:", sr_dir)

    subprocess.run([
        "python", "lpips_measure.py",
        str(gt_dir),
        str(sr_dir)
    ], check=True)


=== Threshold t000 ===
GT dir: /kaggle/working/SAMPLE/DIV2K_train_HR_sample
SR dir: /kaggle/working/srot-proj/SROT-main/results/ESRGAN-SROT-M1234-v2-4x_t000/DIV2K_train_HR


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth
  3%|▎         | 7.88M/233M [00:00<00:02, 81.8MB/s]

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [on]


100%|██████████| 233M/233M [00:02<00:00, 81.7MB/s] 


Loading model from: /usr/local/lib/python3.11/dist-packages/lpips/weights/v0.1/alex.pth
[1/20] 0353.png - LPIPS: 0.2153
[2/20] 0539.png - LPIPS: 0.0815
[3/20] 0483.png - LPIPS: 0.1406
[4/20] 0534.png - LPIPS: 0.2657
[5/20] 0077.png - LPIPS: 0.2706
[6/20] 0020.png - LPIPS: 0.1571
[7/20] 0074.png - LPIPS: 0.2696
[8/20] 0746.png - LPIPS: 0.2519
[9/20] 0558.png - LPIPS: 0.1941
[10/20] 0267.png - LPIPS: 0.1378
[11/20] 0526.png - LPIPS: 0.4896
[12/20] 0040.png - LPIPS: 0.3511
[13/20] 0389.png - LPIPS: 0.1379
[14/20] 0777.png - LPIPS: 0.1978
[15/20] 0256.png - LPIPS: 0.1111
[16/20] 0710.png - LPIPS: 0.4103
[17/20] 0314.png - LPIPS: 0.2179
[18/20] 0328.png - LPIPS: 0.3713
[19/20] 0583.png - LPIPS: 0.1684
[20/20] 0214.png - LPIPS: 0.4074

=== Threshold t010 ===
GT dir: /kaggle/working/SAMPLE/DIV2K_train_HR_sample
SR dir: /kaggle/working/srot-proj/SROT-main/results/ESRGAN-SROT-M1234-v2-4x_t010/DIV2K_train_HR


...

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [on]
Loading model from: /usr/local/lib/python3.11/dist-packages/lpips/weights/v0.1/alex.pth
[1/20] 0353.png - LPIPS: 0.0865
[2/20] 0539.png - LPIPS: 0.0356
[3/20] 0483.png - LPIPS: 0.0696
[4/20] 0534.png - LPIPS: 0.1082
[5/20] 0077.png - LPIPS: 0.0727
[6/20] 0020.png - LPIPS: 0.0539
[7/20] 0074.png - LPIPS: 0.1155
[8/20] 0746.png - LPIPS: 0.0565
[9/20] 0558.png - LPIPS: 0.0730
[10/20] 0267.png - LPIPS: 0.0452
[11/20] 0526.png - LPIPS: 0.2508
[12/20] 0040.png - LPIPS: 0.0537
[13/20] 0389.png - LPIPS: 0.0454
[14/20] 0777.png - LPIPS: 0.0414
[15/20] 0256.png - LPIPS: 0.0646
[16/20] 0710.png - LPIPS: 0.1405
[17/20] 0314.png - LPIPS: 0.0486
[18/20] 0328.png - LPIPS: 0.1396
[19/20] 0583.png - LPIPS: 0.1107
[20/20] 0214.png - LPIPS: 0.1617

=== Threshold t090 ===
GT dir: /kaggle/working/SAMPLE/DIV2K_train_HR_sample
SR dir: /kaggle/working/srot-proj/SROT-main/results/ESRGAN-SROT-M1234-v2-4x_t090/DIV2K_train_HR


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [on]
Loading model from: /usr/local/lib/python3.11/dist-packages/lpips/weights/v0.1/alex.pth
[1/20] 0353.png - LPIPS: 0.0859
[2/20] 0539.png - LPIPS: 0.0354
[3/20] 0483.png - LPIPS: 0.0708
[4/20] 0534.png - LPIPS: 0.1086
[5/20] 0077.png - LPIPS: 0.0801
[6/20] 0020.png - LPIPS: 0.0511
[7/20] 0074.png - LPIPS: 0.1145
[8/20] 0746.png - LPIPS: 0.0573
[9/20] 0558.png - LPIPS: 0.0735
[10/20] 0267.png - LPIPS: 0.0439
[11/20] 0526.png - LPIPS: 0.2425
[12/20] 0040.png - LPIPS: 0.0542
[13/20] 0389.png - LPIPS: 0.0450
[14/20] 0777.png - LPIPS: 0.0414
[15/20] 0256.png - LPIPS: 0.0681
[16/20] 0710.png - LPIPS: 0.1387
[17/20] 0314.png - LPIPS: 0.0488
[18/20] 0328.png - LPIPS: 0.1392
[19/20] 0583.png - LPIPS: 0.1124
[20/20] 0214.png - LPIPS: 0.1575

=== Threshold t100 ===
GT dir: /kaggle/working/SAMPLE/DIV2K_train_HR_sample
SR dir: /kaggle/working/srot-proj/SROT-main/results/ESRGAN-SROT-M1234-v2-4x_t100/DIV2K_train_HR


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [on]
Loading model from: /usr/local/lib/python3.11/dist-packages/lpips/weights/v0.1/alex.pth
[1/20] 0353.png - LPIPS: 0.0862
[2/20] 0539.png - LPIPS: 0.0358
[3/20] 0483.png - LPIPS: 0.0724
[4/20] 0534.png - LPIPS: 0.1100
[5/20] 0077.png - LPIPS: 0.0878
[6/20] 0020.png - LPIPS: 0.0495
[7/20] 0074.png - LPIPS: 0.1149
[8/20] 0746.png - LPIPS: 0.0582
[9/20] 0558.png - LPIPS: 0.0749
[10/20] 0267.png - LPIPS: 0.0432
[11/20] 0526.png - LPIPS: 0.2372
[12/20] 0040.png - LPIPS: 0.0563
[13/20] 0389.png - LPIPS: 0.0454
[14/20] 0777.png - LPIPS: 0.0418
[15/20] 0256.png - LPIPS: 0.0721
[16/20] 0710.png - LPIPS: 0.1408
[17/20] 0314.png - LPIPS: 0.0498
[18/20] 0328.png - LPIPS: 0.1403
[19/20] 0583.png - LPIPS: 0.1141
[20/20] 0214.png - LPIPS: 0.1559


## 4. Generating T_OOS_Maps

In [38]:
import subprocess
from pathlib import Path

# 1) Set up your paths
gt_dir = Path("/kaggle/working/SAMPLE/DIV2K_train_HR_sample")
sr_dir = Path("/kaggle/working/srot-proj/SROT-main/results")

# 2) Build and run the command
cmd = [
    "python", "-u",                       # ← add -u
    "generate_T_OOS_Map.py",
    "-gt", str(gt_dir),
    "-sr", str(sr_dir),
    "--t_num", "11"
]

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

Running: python -u generate_T_OOS_Map.py -gt /kaggle/working/SAMPLE/DIV2K_train_HR_sample -sr /kaggle/working/srot-proj/SROT-main/results --t_num 11
i_idx 0 of 20
i_idx 1 of 20
i_idx 2 of 20
i_idx 3 of 20
i_idx 4 of 20
i_idx 5 of 20
i_idx 6 of 20
i_idx 7 of 20
i_idx 8 of 20
i_idx 9 of 20
i_idx 10 of 20
i_idx 11 of 20
i_idx 12 of 20
i_idx 13 of 20
i_idx 14 of 20
i_idx 15 of 20
i_idx 16 of 20
i_idx 17 of 20
i_idx 18 of 20
i_idx 19 of 20


CompletedProcess(args=['python', '-u', 'generate_T_OOS_Map.py', '-gt', '/kaggle/working/SAMPLE/DIV2K_train_HR_sample', '-sr', '/kaggle/working/srot-proj/SROT-main/results', '--t_num', '11'], returncode=0)

## 5. Training SROOE

In [39]:
### Printing out SROOE train.yml file

srooe_file_path = '/kaggle/working/srooe-proj/SROOE-main/codes/options/train/train.yml'

# Read and print the YAML content
with open(srooe_file_path, 'r') as f:
    srooe_train_data = yaml.safe_load(f)
    print('Train yaml file of SROOE project:\n')
    pprint.pprint(srooe_train_data)

Train yaml file of SROOE project:

{'datasets': {'train': {'GT_size': 256,
                        'batch_size': 8,
                        'color': 'RGB',
                        'dataroot_GT': 'F:\\tempE\\dataset\\Flickr2K_DIV2K_HR_sub',
                        'dataroot_LQ': 'F:\\tempE\\dataset\\Flickr2K_DIV2K_train_LR_bicubic\\X4_sub_120',
                        'dataroot_T_OOS_map': 'E:\\util\\FxSR-PD-LPIPS_21BC_train_best_idx_DF2K_dn_sub',
                        'mode': 'LQGT',
                        'n_workers': 6,
                        'name': 'DF2K',
                        'use_flip': True,
                        'use_rot': True,
                        'use_shuffle': True},
              'val': {'dataroot_GT': 'E:\\exp\\dataset\\DIV2K_valid_HR',
                      'dataroot_LQ': 'E:\\exp\\dataset\\DIV2K_valid_LR_bicubic\\X4',
                      'mode': 'LQGT',
                      'name': 'DIV2K_val_Q100'}},
 'distortion': 'sr',
 'gpu_ids': [0],
 'logger': {'pri

### Modifying SROOE train.yml file

#### Modifying dataset train paths

In [40]:
srooe_train_data['datasets']['train']['dataroot_GT'] = '/kaggle/input/df2kdata/DF2K_train_HR'
srooe_train_data['datasets']['train']['dataroot_LQ'] = '/kaggle/input/df2kdata/DF2K_train_LR_bicubic/X4'
srooe_train_data['datasets']['train']['dataroot_T_OOS_map'] = '/kaggle/working/srot-proj/SROT-main/LPIPS-Map-Gen/T_OOS_Map/ESRGAN-SROT-M1234-v2-4x'

print(f"The dataroot_GT path has been set to {srooe_train_data['datasets']['train']['dataroot_GT']}")
print(f"The dataroot_LQ path has been set to {srooe_train_data['datasets']['train']['dataroot_LQ']}")
print(f"The dataroot_T_OOS_map path has been set to {srooe_train_data['datasets']['train']['dataroot_T_OOS_map']}")

The dataroot_GT path has been set to /kaggle/input/df2kdata/DF2K_train_HR
The dataroot_LQ path has been set to /kaggle/input/df2kdata/DF2K_train_LR_bicubic/X4
The dataroot_T_OOS_map path has been set to /kaggle/working/srot-proj/SROT-main/LPIPS-Map-Gen/T_OOS_Map/ESRGAN-SROT-M1234-v2-4x


#### Modifying dataset validation paths

In [41]:
%cd /kaggle/working

/kaggle/working


In [42]:
!wget -q https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_LR_bicubic_X4.zip
!wget -q https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip

In [43]:
!pwd

/kaggle/working


In [44]:
!mkdir -p DIV2K_val_LRx4
!mkdir -p DIV2K_val_HR

In [45]:
# Unzip into a clean folder
!unzip -q DIV2K_valid_LR_bicubic_X4.zip -d DIV2K_val_LRx4
!unzip -q DIV2K_valid_HR.zip -d DIV2K_val_HR

In [46]:
!ls '/kaggle/working/DIV2K_val_LRx4/DIV2K_valid_LR_bicubic/X4'

0801x4.png  0818x4.png	0835x4.png  0852x4.png	0869x4.png  0886x4.png
0802x4.png  0819x4.png	0836x4.png  0853x4.png	0870x4.png  0887x4.png
0803x4.png  0820x4.png	0837x4.png  0854x4.png	0871x4.png  0888x4.png
0804x4.png  0821x4.png	0838x4.png  0855x4.png	0872x4.png  0889x4.png
0805x4.png  0822x4.png	0839x4.png  0856x4.png	0873x4.png  0890x4.png
0806x4.png  0823x4.png	0840x4.png  0857x4.png	0874x4.png  0891x4.png
0807x4.png  0824x4.png	0841x4.png  0858x4.png	0875x4.png  0892x4.png
0808x4.png  0825x4.png	0842x4.png  0859x4.png	0876x4.png  0893x4.png
0809x4.png  0826x4.png	0843x4.png  0860x4.png	0877x4.png  0894x4.png
0810x4.png  0827x4.png	0844x4.png  0861x4.png	0878x4.png  0895x4.png
0811x4.png  0828x4.png	0845x4.png  0862x4.png	0879x4.png  0896x4.png
0812x4.png  0829x4.png	0846x4.png  0863x4.png	0880x4.png  0897x4.png
0813x4.png  0830x4.png	0847x4.png  0864x4.png	0881x4.png  0898x4.png
0814x4.png  0831x4.png	0848x4.png  0865x4.png	0882x4.png  0899x4.png
0815x4.png  0832x4.png	0849x4.png 

In [47]:
!ls '/kaggle/working/DIV2K_val_HR/DIV2K_valid_HR'

0801.png  0814.png  0827.png  0840.png	0853.png  0866.png  0879.png  0892.png
0802.png  0815.png  0828.png  0841.png	0854.png  0867.png  0880.png  0893.png
0803.png  0816.png  0829.png  0842.png	0855.png  0868.png  0881.png  0894.png
0804.png  0817.png  0830.png  0843.png	0856.png  0869.png  0882.png  0895.png
0805.png  0818.png  0831.png  0844.png	0857.png  0870.png  0883.png  0896.png
0806.png  0819.png  0832.png  0845.png	0858.png  0871.png  0884.png  0897.png
0807.png  0820.png  0833.png  0846.png	0859.png  0872.png  0885.png  0898.png
0808.png  0821.png  0834.png  0847.png	0860.png  0873.png  0886.png  0899.png
0809.png  0822.png  0835.png  0848.png	0861.png  0874.png  0887.png  0900.png
0810.png  0823.png  0836.png  0849.png	0862.png  0875.png  0888.png
0811.png  0824.png  0837.png  0850.png	0863.png  0876.png  0889.png
0812.png  0825.png  0838.png  0851.png	0864.png  0877.png  0890.png
0813.png  0826.png  0839.png  0852.png	0865.png  0878.png  0891.png


In [48]:
srooe_train_data['datasets']['val']['dataroot_GT'] = '/kaggle/working/DIV2K_val_HR/DIV2K_valid_HR'
srooe_train_data['datasets']['val']['dataroot_LQ'] = '/kaggle/working/DIV2K_val_LRx4/DIV2K_valid_LR_bicubic/X4'

### Modifying pretrain_model_G generator model path in train.yml file

In [49]:
srooe_train_data['path']['pretrain_model_G'] = pre_trained_SROT_path
print(f"The path has been set to {srooe_train_data['path']['pretrain_model_G']}")

The path has been set to /kaggle/working/srot-chck-pth/SR.pth


### Changing the niter to limit training epochs in train.yml


In [50]:
if N_ITER is not None:
    srooe_train_data['train']['niter'] = N_ITER

### Changing val_frequency in train.yml

In [51]:
if VAL_FREQUENCY is not None:
    srooe_train_data['train']['val_freq'] = VAL_FREQUENCY

### Dumping modified srooe_train_data back into SROOE train.yml file

In [53]:
with open(srooe_file_path, 'w') as f:
    yaml.dump(srooe_train_data, f, sort_keys=False)
    print('Train yaml file of SROOE project updated:\n')
    pprint.pprint(srooe_train_data)

Train yaml file of SROOE project updated:

{'datasets': {'train': {'GT_size': 256,
                        'batch_size': 8,
                        'color': 'RGB',
                        'dataroot_GT': '/kaggle/input/df2kdata/DF2K_train_HR',
                        'dataroot_LQ': '/kaggle/input/df2kdata/DF2K_train_LR_bicubic/X4',
                        'dataroot_T_OOS_map': '/kaggle/working/srot-proj/SROT-main/LPIPS-Map-Gen/T_OOS_Map/ESRGAN-SROT-M1234-v2-4x',
                        'mode': 'LQGT',
                        'n_workers': 6,
                        'name': 'DF2K',
                        'use_flip': True,
                        'use_rot': True,
                        'use_shuffle': True},
              'val': {'dataroot_GT': '/kaggle/working/DIV2K_val_HR/DIV2K_valid_HR',
                      'dataroot_LQ': '/kaggle/working/DIV2K_val_LRx4/DIV2K_valid_LR_bicubic/X4',
                      'mode': 'LQGT',
                      'name': 'DIV2K_val_Q100'}},
 'distortion': '

## 6. SROOE train.py run

### Changing current working dir

In [54]:
%cd /kaggle/working/srooe-proj/SROOE-main/codes

/kaggle/working/srooe-proj/SROOE-main/codes


In [55]:
!pwd

/kaggle/working/srooe-proj/SROOE-main/codes


In [56]:
!ls

data  models  options  PerceptualSimilarity  scripts  test.py  train.py  utils


### Running train.py file of SROOE project

In [57]:
!pip install --quiet lmdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.6/299.6 kB 8.8 MB/s eta 0:00:00


In [58]:
# Path to your training script and config
train_script = Path("train.py")
train_opt    = Path("options") / "train" / "train.yml"

# Build and run the command
cmd = [
    "python","-u",
    str(train_script),
    "-opt",
    str(train_opt)
]

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

Running: python -u train.py -opt options/train/train.yml
export CUDA_VISIBLE_DEVICES=0
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /usr/local/lib/python3.11/dist-packages/lpips/weights/v0.1/alex.pth
Disabled distributed training.


25-08-09 05:10:41.295 - INFO:   name: SROOE-UNet-M1234-v2
  use_tb_logger: True
  model: srgan
  distortion: sr
  scale: 4
  gpu_ids: [0]
  datasets:[
    train:[
      name: DF2K
      mode: LQGT
      dataroot_GT: /kaggle/input/df2kdata/DF2K_train_HR
      dataroot_LQ: /kaggle/input/df2kdata/DF2K_train_LR_bicubic/X4
      dataroot_T_OOS_map: /kaggle/working/srot-proj/SROT-main/LPIPS-Map-Gen/T_OOS_Map/ESRGAN-SROT-M1234-v2-4x
      use_shuffle: True
      n_workers: 6
      batch_size: 8
      GT_size: 256
      use_flip: True
      use_rot: True
      color: RGB
      phase: train
      scale: 4
      data_type: img
    ]
    val:[
      name: DIV2K_val_Q100
      mode: LQGT
      dataroot_GT: /kaggle/working/DIV2K_val_HR/DIV2K_valid_HR
      dataroot_LQ: /kaggle/working/DIV2K_val_LRx4/DIV2K_valid_LR_bicubic/X4
      phase: val
      scale: 4
      data_type: img
    ]
  ]
  network_G:[
    which_model_G: RRDBNet
    in_nc: 4
    out_nc: 3
    nf: 64
    nb: 23
    scale: 4
  ]
  netw

paths_GT: 20
paths_LQ: 20
paths_LPIPS_map: 60
Chosen subset size: 20
we have gone too far!
we have gone too far! 2
Setting up Perceptual loss...


...

we have gone too far! 4
Tensor img_LQ shape: torch.Size([3, 339, 510])
Tensor img_GT shape: torch.Size([3, 1356, 2040])
Tensor img_GT_LPIPS_map shape: torch.Size([3, 339, 510])
Tensor img_LQ shape: torch.Size([3, 339, 510])
Tensor img_GT shape: torch.Size([3, 1356, 2040])
Tensor img_GT_LPIPS_map shape: torch.Size([3, 339, 510])
Tensor img_LQ shape: torch.Size([3, 384, 510])
Tensor img_GT shape: torch.Size([3, 1536, 2040])
Tensor img_GT_LPIPS_map shape: torch.Size([3, 384, 510])
Tensor img_LQ shape: torch.Size([3, 300, 510])
Tensor img_GT shape: torch.Size([3, 1200, 2040])
Tensor img_GT_LPIPS_map shape: torch.Size([3, 300, 510])
Tensor img_LQ shape: torch.Size([3, 384, 510])
Tensor img_GT shape: torch.Size([3, 1536, 2040])
Tensor img_GT_LPIPS_map shape: torch.Size([3, 384, 510])
Tensor img_LQ shape: torch.Size([3, 339, 510])
Tensor img_GT shape: torch.Size([3, 1356, 2040])
Tensor img_GT_LPIPS_map shape: torch.Size([3, 339, 510])
Tensor img_LQ shape: torch.Size([3, 282, 510])
Tensor img_

25-08-09 05:25:54.418 - INFO: # Validation # PSNR: 2.7630e+01
25-08-09 05:25:54.418 - INFO: # Validation # LPIPS: 0.099546
25-08-09 05:25:54.418 - INFO: <epoch: 99, iter:     200> psnr: 2.7630e+01 lpips: 0.099546


100 of 100
Tensor img_LQ shape: torch.Size([3, 64, 64])
Tensor img_GT shape: torch.Size([3, 256, 256])
Tensor img_GT_LPIPS_map shape: torch.Size([3, 64, 64])
Tensor img_LQ shape: torch.Size([3, 64, 64])
Tensor img_GT shape: torch.Size([3, 256, 256])
Tensor img_GT_LPIPS_map shape: torch.Size([3, 64, 64])
Tensor img_LQ shape: torch.Size([3, 64, 64])
Tensor img_GT shape: torch.Size([3, 256, 256])
Tensor img_GT_LPIPS_map shape: torch.Size([3, 64, 64])
Tensor img_LQ shape: torch.Size([3, 64, 64])
Tensor img_GT shape: torch.Size([3, 256, 256])
Tensor img_GT_LPIPS_map shape: torch.Size([3, 64, 64])
Tensor img_LQ shape: torch.Size([3, 64, 64])
Tensor img_GT shape: torch.Size([3, 256, 256])
Tensor img_GT_LPIPS_map shape: torch.Size([3, 64, 64])
Tensor img_LQ shape: torch.Size([3, 64, 64])
Tensor img_GT shape: torch.Size([3, 256, 256])
Tensor img_GT_LPIPS_map shape: torch.Size([3, 64, 64])
Tensor img_LQ shape: torch.Size([3, 64, 64])
Tensor img_GT shape: torch.Size([3, 256, 256])
Tensor img_GT_L

25-08-09 05:25:58.158 - INFO: Saving the final model.
25-08-09 05:25:58.378 - INFO: End of training.


CompletedProcess(args=['python', '-u', 'train.py', '-opt', 'options/train/train.yml'], returncode=0)